# Spectral Surgery post-hoc: confirmation v3, 4 tasks x 6 seeds

Qwen2.5-1.5B-Instruct, SST-2 / QNLI / MNLI-m / MNLI-mm, seeds **6101-6106**: **24 jobs**, only `spectral_surgery_posthoc`.

Uses the existing v3 splits, rank [2,4,8,4], q_proj/v_proj, 24/48/48 calibration/gate/monitor examples, 8+32 returns, and 1024 evaluation examples per task. Spectral edits the final freshness-trained adapter once using answer-token NLL including EOS. This is an adapted post-hoc baseline on the existing confirmation data, not an original-paper benchmark reproduction.

Enable **Internet** and select **GPU T4 x2**, then Run All. Two independent jobs run at once, one per GPU. Results are separate from the existing RIFT board. Harmful rates measure training **before** the final edit; compare pre/post edit Accuracy and class NLL to assess the edit itself.


In [ ]:
from pathlib import Path
import json
import os
import signal
import subprocess
import sys
import time
import zipfile
from IPython.display import display, Markdown, FileLink

REPO_URL = "https://github.com/TrgPhan/VASTLoRA.git"
REPO_REF = "8dabdea2cad7e8fdaf61ae8993329f7209a99682"
WORK_ROOT = Path("/kaggle/working")
REPO_DIR = WORK_ROOT / ("RIFTLoRA-spectral-" + REPO_REF[:8])
OUTPUT_ROOT = WORK_ROOT / "spectral_posthoc_v3_supplement"
GPU_IDS = [0, 1]
MAX_JOBS = None  # None: all missing jobs; a positive integer caps NEW jobs this session.
RESUME_ROOTS = []  # e.g. [Path("/kaggle/input/my-results/spectral_posthoc_v3_supplement")]
WORK_ROOT.mkdir(parents=True, exist_ok=True)
print({"method": "spectral_surgery_posthoc", "total_jobs": 24,
       "seeds": list(range(6101, 6107)), "repo_commit": REPO_REF})


In [ ]:
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run(["git", "checkout", "--detach", REPO_REF], cwd=REPO_DIR, check=True)
resolved = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
if resolved != REPO_REF:
    raise RuntimeError(f"Unexpected checkout {resolved}; choose a separate REPO_DIR.")
dirty = subprocess.check_output(
    ["git", "status", "--porcelain", "--untracked-files=no"], cwd=REPO_DIR, text=True
).strip()
if dirty:
    raise RuntimeError(f"Tracked changes in checkout: {dirty}")
if not (REPO_DIR / "scripts/run_spectral_v3_confirmation.py").exists():
    raise RuntimeError("Pinned runner is missing from GitHub checkout.")

# Pin the HF stack tested locally; preserve Kaggle's installed CUDA Torch build.
import importlib.metadata as metadata
constraints = WORK_ROOT / "spectral-torch-constraint.txt"
constraints.write_text("torch==" + metadata.version("torch") + "\n", encoding="utf-8")
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "-c", str(constraints), "-e", ".[scale,dev]",
    "transformers==5.10.2", "peft==0.20.0", "accelerate==1.14.0",
    "bitsandbytes==0.50.2", "datasets==5.0.0",
], cwd=REPO_DIR, check=True)
print("Installed implementation:", resolved)


In [ ]:
subprocess.run(["nvidia-smi"], check=True)
import torch
if len(set(GPU_IDS)) != len(GPU_IDS) or any(g < 0 or g >= torch.cuda.device_count() for g in GPU_IDS):
    raise RuntimeError(f"Requested {GPU_IDS}; available GPUs: {torch.cuda.device_count()}")
for gpu in GPU_IDS:
    props = torch.cuda.get_device_properties(gpu)
    print({"gpu": gpu, "name": props.name, "memory_gib": round(props.total_memory / 2**30, 2)})

RUNNER = REPO_DIR / "scripts/run_spectral_v3_confirmation.py"
subprocess.run([
    sys.executable, "-m", "pytest", "-q",
    "tests/test_spectral_v3_confirmation.py",
    "tests/test_paper_baseline_integration.py",
    "tests/test_spectral_surgery.py",
], cwd=REPO_DIR, check=True)
preflight = subprocess.check_output([
    sys.executable, str(RUNNER), "--output-root", str(OUTPUT_ROOT), "--dry-run",
], cwd=REPO_DIR, text=True)
plan = json.loads(preflight)
assert plan["jobs"] == 24 and plan["seeds"] == list(range(6101, 6107))
display({"jobs": plan["jobs"], "tasks": plan["tasks"], "seeds": plan["seeds"],
         "model": plan["example_config"]["model"],
         "calibration": plan["example_config"]["experiment"]["calibration_gradient_examples"],
         "rank": plan["example_config"]["experiment"]["client_ranks"]})


In [ ]:
# Download each asset once before the two GPU workers start.
from huggingface_hub import snapshot_download
from datasets import load_dataset
base = json.loads((REPO_DIR / "configs/local_1_5b_rift_development.json").read_text())
snapshot_download(base["model"]["name"], revision=base["model"]["revision"])
for subset in ["sst2", "qnli", "mnli"]:
    load_dataset(base["dataset"]["hub_path"], subset, revision=base["dataset"]["revision"])
print("Pinned model and GLUE datasets cached.")


## Run and resume

A fresh session schedules 24 jobs. Valid completed jobs are skipped only when commit, matrix/config fingerprints, seed and result CSVs match. A failed or interrupted job restarts from the beginning; this runner does not checkpoint mid-job.

To continue in a later Kaggle session, save outputs as a Dataset, attach it, and set `RESUME_ROOTS` to the directory containing the task folders. Use outputs from this notebook's pinned commit. Old per-return Spectral and development results are different experiments.

Failure/OOM stops active workers and prints the error. No automatic config reductions are applied. After an interruption, run the summary and archive cells manually to collect completed results. With `MAX_JOBS` set, the tables remain explicitly partial.


In [ ]:
command = [sys.executable, "-u", str(RUNNER), "--output-root", str(OUTPUT_ROOT)]
for gpu in GPU_IDS:
    command.extend(["--gpu", str(gpu)])
for source in RESUME_ROOTS:
    source = Path(source)
    if not source.is_dir():
        raise FileNotFoundError(source)
    command.extend(["--resume-root", str(source)])
if MAX_JOBS is not None:
    command.extend(["--max-jobs", str(MAX_JOBS)])

process = subprocess.Popen(
    command, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, start_new_session=True,
)
try:
    for line in process.stdout:
        print(line.rstrip(), flush=True)
    return_code = process.wait()
    if return_code:
        raise RuntimeError(f"Runner failed (exit {return_code}); inspect printed log. Completed results remain available.")
finally:
    if process.poll() is None:
        os.killpg(process.pid, signal.SIGINT)
        try:
            process.wait(timeout=30)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid, signal.SIGKILL)
            process.wait()
    process.stdout.close()


In [ ]:
# Also runnable after an interrupted training cell once its workers have stopped.
subprocess.run([
    sys.executable, str(RUNNER), "--output-root", str(OUTPUT_ROOT), "--summarize-only",
], cwd=REPO_DIR, check=True)
status = json.loads((OUTPUT_ROOT / "completion.json").read_text())
display(status)
if status["completed"]:
    import pandas as pd
    runs = pd.read_csv(OUTPUT_ROOT / "runs.csv")
    summary = pd.read_csv(OUTPUT_ROOT / "summary.csv")
    for title, columns in [
        ("Accuracy (%)", ["seeds", "accuracy_mean_pct", "accuracy_sd_pct", "edit_accuracy_delta_pp"]),
        ("Class NLL", ["seeds", "class_nll_mean", "class_nll_sd", "edit_class_nll_delta"]),
        ("Harmful (%) before post-hoc edit", ["seeds", "harmful_mean_pct", "late_harmful_mean_pct"]),
    ]:
        display(Markdown("### " + title))
        display(summary[["task", *columns]])
    display(Markdown("### All completed seeds, including pre/post edit quality"))
    display(runs)
else:
    print("No validated results yet; no metrics have been fabricated.")


In [ ]:
archive = WORK_ROOT / ("spectral-posthoc-v3-" + str(time.time_ns()) + ".zip")
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(OUTPUT_ROOT.rglob("*")):
        if path.is_file() and path.name != ".launcher.lock":
            bundle.write(path, Path(OUTPUT_ROOT.name) / path.relative_to(OUTPUT_ROOT))
print("Results archive:", archive, "MiB:", round(archive.stat().st_size / 2**20, 2))
display(FileLink(str(archive)))
